In [2]:
import sqlite3
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Connect to the database
try:
    conn = sqlite3.connect('gpt56_terra_baseline.db')
    # conn = sqlite3.connect('qwen36_27b_baseline.db')
    df = pd.read_sql_query("SELECT label_truth, guess FROM text_guess", conn)
    conn.close()

    # Standardize to lowercase
    df['guess_lower'] = df['guess'].str.lower().str.strip()
    df['truth_lower'] = df['label_truth'].str.lower().str.strip()

    # 1. Check if all guesses are 'ai' or 'human'
    unique_guesses = df['guess_lower'].unique()
    invalid_guesses = [g for g in unique_guesses if g not in ['ai', 'human']]

    if len(invalid_guesses) == 0:
        check_msg = "Check passed: All guesses are either 'ai' or 'human'."
        
        # Map to binary for sklearn (AI = 1, Human = 0)
        y_true = (df['truth_lower'] == 'ai').astype(int)
        y_pred = (df['guess_lower'] == 'ai').astype(int)
        
        # Calculate metrics
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        
        metrics_msg = (
            f"Accuracy: {acc:.4f}\n"
            f"Precision: {prec:.4f}\n"
            f"Recall: {rec:.4f}\n"
            f"F1 Score: {f1:.4f}"
        )
        print(f"{check_msg}\n{metrics_msg}")
    else:
        print(f"Check failed: Found invalid guesses: {invalid_guesses}")

except Exception as e:
    print(f"Error: {e}")

Check passed: All guesses are either 'ai' or 'human'.
Accuracy: 0.7640
Precision: 0.9312
Recall: 0.5644
F1 Score: 0.7028


You are evaluating a text to determine if it was written by an AI or a Human.

Text:
Is there actually a subreddit for real computer science discussion? I keep running into places that are mostly beginner programming questions, hardware help, and the same stuff you’d expect from high school students just getting started. Nothing wrong with that, but I’m looking for something more focused on the actual theory, research, and deeper CS topics. If anyone knows a good one, I’d appreciate it.

Explanation hint:
- Look for natural, conversational phrasing like 'I keep running into'.
- Look for conversational hedges and qualifiers like 'mostly', 'just', 'actually'.
- Look for authentic, personal frustration with existing online community standards.
- Look for common contractions like 'I’m', 'don’t', 'it’s'.

Based on the text and the explanation, make a decision. 
Respond with EXACTLY one word: either "AI" or "Human".